<a href="https://colab.research.google.com/github/priyal6/DL/blob/main/ATTN_AND_FLASH.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn.functional as F

def standard_attention(Q, K, V):
    """
    Q, K, V: (batch, seq_len, d)
    """
    d = Q.size(-1)

    # creates a huge (N × N) matrix
    scores = torch.matmul(Q, K.transpose(-2, -1)) / (d ** 0.5)

    attn = F.softmax(scores, dim=-1)

    output = torch.matmul(attn, V)

    return output

In [8]:
def flash_attention(Q, K, V, block_size=64):
    """
    Simplified FlashAttention
    Q, K, V: (batch, seq_len, d)
    """
    B, N, d = Q.shape

    output = torch.zeros_like(Q)

    scale = d ** -0.5

    for b in range(B):
        for i in range(0, N, block_size):
          # Instead of all tokens, we handle small chunks.
            Q_block = Q[b, i:i+block_size]


            m_i = torch.full((Q_block.size(0),), float('-inf'), device=Q.device)
            l_i = torch.zeros(Q_block.size(0), device=Q.device)
            O_i = torch.zeros_like(Q_block)

            for j in range(0, N, block_size):
                K_block = K[b, j:j+block_size]
                V_block = V[b, j:j+block_size]


                S_ij = torch.matmul(Q_block, K_block.transpose(0, 1)) * scale


                m_ij = torch.max(S_ij, dim=1).values


                m_new = torch.maximum(m_i, m_ij)


                exp_ij = torch.exp(S_ij - m_new.unsqueeze(1))


                l_i = torch.exp(m_i - m_new) * l_i + exp_ij.sum(dim=1)


                O_i = (
                    torch.exp(m_i - m_new).unsqueeze(1) * O_i +
                    torch.matmul(exp_ij, V_block)
                )

                m_i = m_new


            O_i = O_i / l_i.unsqueeze(1)

            output[b, i:i+block_size] = O_i

    return output

In [10]:
# For each block:
#     compute partial scores
#     update running softmax
#     accumulate output

In [9]:

torch.manual_seed(0)

B, N, d = 1, 128, 64

Q = torch.randn(B, N, d)
K = torch.randn(B, N, d)
V = torch.randn(B, N, d)


out_standard = standard_attention(Q, K, V)
out_flash = flash_attention(Q, K, V, block_size=32)

# Compare
print("Max difference:", torch.max(torch.abs(out_standard - out_flash)))

Max difference: tensor(2.9802e-07)
